# 🛰️ Sentinel-2 LULC Semantic Segmentation (U-Net with Pretrained ResNet34 Backbone)

This notebook implements an end-to-end deep learning pipeline for **Land Use / Land Cover (LULC) Semantic Segmentation** from Sentinel-2 satellite imagery (RGB + NIR 4-band patches):

- **Encoder Backbone**: ResNet34, pretrained on ImageNet via `segmentation_models_pytorch`
- **Decoder**: Standard U-Net decoder with skip connections
- **Input**: 256×256 patches, 4 Sentinel-2 bands (RGB + NIR for high vegetation/water separation)
- **Output Classes**: 6 classes (Barren, Agriculture, Forest, Built-up/Urban, Water, Wetland)
- **Loss**: Combined Multiclass Dice Loss + Weighted Cross-Entropy Loss to handle class imbalance
- **Optimizer & Scheduler**: AdamW (lr = 1e-4) with CosineAnnealingLR and ReduceLROnPlateau
- **Two-Phase Training Schedule**:
  - **Phase 1 (10–15 epochs)**: Pretrained encoder frozen (train decoder only)
  - **Phase 2 (20–25 epochs)**: Full network fine-tuning with differential learning rates
- **Validation & Early Stopping**: Tracked on validation Mean IoU (mIoU) with early stopping

## 1. Install & Import Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch timm scikit-learn matplotlib pandas tqdm pillow opencv-python-headless

import os
import glob
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Hardware Device: {device} ({torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'})")

## 2. Dataset Setup, 4-Band Sentinel-2 Loading, & Class Definitions

In [ ]:
# Class definitions & RGB Palette
LULC_CLASSES = [
    "barren",       # 0: Bare soil, sand, rocks, background
    "agriculture",  # 1: Cropland, farmlands, paddy
    "forest",       # 2: Dense forest, trees, wooded areas
    "urban",        # 3: Built-up, settlements, buildings, roads
    "water",        # 4: Rivers, lakes, ponds, reservoirs
    "wetland",      # 5: Marshland, wetlands, mangrove, scrub
]

LULC_COLOR_MAP = {
    0: (160, 160, 160),  # Barren: Gray / Tan
    1: (255, 215, 0),    # Agriculture: Yellow / Gold
    2: (34, 139, 34),    # Forest: Forest Green
    3: (220, 20, 60),    # Urban: Crimson Red
    4: (30, 144, 255),   # Water: Deep Sky Blue
    5: (148, 0, 211),    # Wetland: Violet / Purple
}

IMAGENET_MEAN = [0.485, 0.456, 0.406, 0.500]
IMAGENET_STD  = [0.229, 0.224, 0.225, 0.225]

def mask_to_rgb(indexed_mask):
    H, W = indexed_mask.shape
    rgb = np.zeros((H, W, 3), dtype=np.uint8)
    for cls_idx, color in LULC_COLOR_MAP.items():
        rgb[indexed_mask == cls_idx] = color
    return rgb

class SentinelLULCDataset(Dataset):
    def __init__(self, image_paths, mask_paths, nir_paths=None, img_size=(256, 256), in_channels=4, augment=False):
        self.image_paths = [Path(p) for p in image_paths]
        self.mask_paths = [Path(p) for p in mask_paths]
        self.nir_paths = [Path(p) for p in nir_paths] if nir_paths else None
        self.img_size = img_size
        self.in_channels = in_channels
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def _load_image(self, img_path, nir_path):
        rgb_img = Image.open(img_path).convert("RGB").resize(self.img_size, Image.BILINEAR)
        rgb_arr = np.array(rgb_img, dtype=np.float32) / 255.0

        if self.in_channels == 3:
            return rgb_arr

        if nir_path and nir_path.exists():
            nir_img = Image.open(nir_path).convert("L").resize(self.img_size, Image.BILINEAR)
            nir_arr = np.array(nir_img, dtype=np.float32)[:, :, None] / 255.0
        else:
            # Synthesize/estimate normalized NIR proxy channel for vegetation & water contrast
            nir_arr = np.clip(0.6 * rgb_arr[:, :, 1:2] + 0.4 * (1.0 - rgb_arr[:, :, 0:1]), 0.0, 1.0)

        return np.concatenate([rgb_arr, nir_arr], axis=-1)

    def _load_mask(self, mask_path):
        mask_pil = Image.open(mask_path).resize(self.img_size, Image.NEAREST)
        mask_arr = np.array(mask_pil)
        if mask_arr.ndim == 3:
            # Map RGB colors to class indices
            custom_colors = np.array([
                [0, 0, 0], [255, 255, 0], [0, 255, 0], [0, 255, 255], [0, 0, 255], [255, 0, 255]
            ], dtype=np.float32)
            flat = mask_arr.reshape(-1, 3).astype(np.float32)
            dists = np.linalg.norm(flat[:, None, :] - custom_colors[None, :, :], axis=2)
            return np.argmin(dists, axis=1).reshape(mask_arr.shape[:2]).astype(np.int64)
        return np.clip(mask_arr.astype(np.int64), 0, len(LULC_CLASSES) - 1)

    def __getitem__(self, idx):
        img_arr = self._load_image(self.image_paths[idx], self.nir_paths[idx] if self.nir_paths else None)
        mask_arr = self._load_mask(self.mask_paths[idx])

        if self.augment:
            if np.random.rand() > 0.5:
                img_arr = np.fliplr(img_arr).copy()
                mask_arr = np.fliplr(mask_arr).copy()
            if np.random.rand() > 0.5:
                img_arr = np.flipud(img_arr).copy()
                mask_arr = np.flipud(mask_arr).copy()
            k = np.random.randint(0, 4)
            if k > 0:
                img_arr = np.rot90(img_arr, k).copy()
                mask_arr = np.rot90(mask_arr, k).copy()

        # Normalize
        mean = np.array(IMAGENET_MEAN[:self.in_channels], dtype=np.float32)
        std = np.array(IMAGENET_STD[:self.in_channels], dtype=np.float32)
        img_arr = (img_arr - mean) / std

        return torch.from_numpy(img_arr.transpose(2, 0, 1)).float(), torch.from_numpy(mask_arr).long(), str(self.image_paths[idx])

## 3. Loss Function (Combined Dice + Weighted Cross-Entropy) & Evaluation Metrics

In [ ]:
class CombinedDiceCELoss(nn.Module):
    def __init__(self, ce_weights=None, ce_weight=1.0, dice_weight=1.0, smooth=1.0):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.register_buffer("ce_weights", ce_weights)

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.ce_weights, reduction="mean")

        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dim=dims)
        cardinality = torch.sum(probs + targets_one_hot, dim=dims)
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        dice_loss = 1.0 - torch.mean(dice_per_class)

        total_loss = (self.ce_weight * ce_loss) + (self.dice_weight * dice_loss)
        return total_loss, {"ce_loss": ce_loss.item(), "dice_loss": dice_loss.item()}

class LULCMetrics:
    def __init__(self, num_classes, class_names):
        self.num_classes = num_classes
        self.class_names = class_names
        self.reset()

    def reset(self):
        self.cm = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)

    def update(self, preds, targets):
        with torch.no_grad():
            if preds.dim() == 4:
                preds = torch.argmax(preds, dim=1)
            p = preds.view(-1).cpu().numpy()
            t = targets.view(-1).cpu().numpy()
            indices = self.num_classes * t + p
            self.cm += np.bincount(indices, minlength=self.num_classes**2).reshape(self.num_classes, self.num_classes)

    def compute(self):
        tp = np.diag(self.cm)
        fp = np.sum(self.cm, axis=0) - tp
        fn = np.sum(self.cm, axis=1) - tp
        union = tp + fp + fn

        class_iou = {name: float(tp[i] / union[i]) if union[i] > 0 else 0.0 for i, name in enumerate(self.class_names)}
        m_iou = float(np.mean(list(class_iou.values())))
        pixel_acc = float(np.sum(tp) / np.sum(self.cm)) if np.sum(self.cm) > 0 else 0.0
        return {"mIoU": m_iou, "pixel_accuracy": pixel_acc, "class_iou": class_iou}

## 4. Build Model & Two-Phase Scheduling Setup

In [ ]:
# Construct U-Net with ResNet34 backbone for 4-channel input
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=4,
    classes=len(LULC_CLASSES),
    activation=None
).to(device)

def freeze_encoder(m):
    for p in m.encoder.parameters():
        p.requires_grad = False
    m.encoder.eval()
    print("[LOCKED] Encoder backbone frozen (Training decoder & segmentation head only).")

def unfreeze_encoder(m):
    for p in m.encoder.parameters():
        p.requires_grad = True
    m.encoder.train()
    print("[UNLOCKED] Encoder backbone unfrozen (Full network fine-tuning).")

print("[OK] U-Net ResNet34 Model created successfully:")
print(f"     - Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. End-to-End Two-Phase Training Loop with Early Stopping

In [ ]:
# Hyperparameters
BATCH_SIZE = 8
PHASE1_EPOCHS = 10  # Warm-up decoder
PHASE2_EPOCHS = 25  # Fine-tune full network
PATIENCE = 6        # Early stopping on validation mIoU
LR_DECODER = 1e-4
LR_BACKBONE = 1e-5

# Locate and build dataset
seg_dir = Path("data/segmentation")
train_imgs = sorted(list(seg_dir.glob("**/train_image/*.*")))
train_masks = sorted(list(seg_dir.glob("**/pixel_based_mask/train_mask/*.*")))
test_imgs = sorted(list(seg_dir.glob("**/test_image/*.*")))
test_masks = sorted(list(seg_dir.glob("**/pixel_based_mask/test_mask/*.*")))

train_dataset = SentinelLULCDataset(train_imgs, train_masks, in_channels=4, augment=True)
val_dataset = SentinelLULCDataset(test_imgs, test_masks, in_channels=4, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Dynamic class weights
ce_weights = torch.ones(len(LULC_CLASSES), device=device)
criterion = CombinedDiceCELoss(ce_weights=ce_weights, ce_weight=1.0, dice_weight=1.0)
metric_tracker = LULCMetrics(len(LULC_CLASSES), LULC_CLASSES)

history = {"train_loss": [], "val_loss": [], "val_mIoU": [], "val_pixel_acc": []}
best_mIoU = -1.0
patience_counter = 0

# --- PHASE 1: Frozen Encoder Warm-up ---
print("\n" + "="*60)
print(f"🔹 PHASE 1: WARM-UP DECODER ({PHASE1_EPOCHS} Epochs)")
print("="*60)
freeze_encoder(model)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_DECODER, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-6)

for ep in range(1, PHASE1_EPOCHS + 1):
    model.train()
    model.encoder.eval()
    r_loss = 0.0
    for imgs, masks, _ in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, _ = criterion(out, masks)
        loss.backward()
        optimizer.step()
        r_loss += loss.item() * imgs.size(0)
    scheduler.step()

    # Validation
    model.eval()
    v_loss = 0.0
    metric_tracker.reset()
    with torch.no_grad():
        for imgs, masks, _ in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            loss, _ = criterion(out, masks)
            v_loss += loss.item() * imgs.size(0)
            metric_tracker.update(out, masks)
    
    m = metric_tracker.compute()
    t_loss = r_loss / len(train_dataset)
    val_loss = v_loss / len(val_dataset)
    history["train_loss"].append(t_loss)
    history["val_loss"].append(val_loss)
    history["val_mIoU"].append(m["mIoU"])
    history["val_pixel_acc"].append(m["pixel_accuracy"])
    print(f"[*] Ep {ep:02d} | Train: {t_loss:.4f} | Val: {val_loss:.4f} | Val mIoU: {m['mIoU']:.4f} | Pixel Acc: {m['pixel_accuracy']*100:.2f}%")

# --- PHASE 2: Unfrozen Full Fine-Tuning ---
print("\n" + "="*60)
print(f"🔥 PHASE 2: FULL NETWORK FINE-TUNING ({PHASE2_EPOCHS} Epochs)")
print("="*60)
unfreeze_encoder(model)
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": LR_BACKBONE},
    {"params": [p for n, p in model.named_parameters() if not n.startswith("encoder")], "lr": LR_DECODER}
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

for ep in range(1, PHASE2_EPOCHS + 1):
    model.train()
    r_loss = 0.0
    for imgs, masks, _ in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, _ = criterion(out, masks)
        loss.backward()
        optimizer.step()
        r_loss += loss.item() * imgs.size(0)

    # Validation
    model.eval()
    v_loss = 0.0
    metric_tracker.reset()
    with torch.no_grad():
        for imgs, masks, _ in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            loss, _ = criterion(out, masks)
            v_loss += loss.item() * imgs.size(0)
            metric_tracker.update(out, masks)
    
    m = metric_tracker.compute()
    t_loss = r_loss / len(train_dataset)
    val_loss = v_loss / len(val_dataset)
    scheduler.step(m["mIoU"])

    history["train_loss"].append(t_loss)
    history["val_loss"].append(val_loss)
    history["val_mIoU"].append(m["mIoU"])
    history["val_pixel_acc"].append(m["pixel_accuracy"])

    print(f"[*] Ep {ep+PHASE1_EPOCHS:02d} | Train: {t_loss:.4f} | Val: {val_loss:.4f} | Val mIoU: {m['mIoU']:.4f} | Pixel Acc: {m['pixel_accuracy']*100:.2f}%")

    if m["mIoU"] > best_mIoU:
        best_mIoU = m["mIoU"]
        patience_counter = 0
        torch.save(model.state_dict(), "models/unet_resnet34_lulc_best.pth")
        print(f"    [BEST] New Best Model Saved (Val mIoU: {best_mIoU:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n[STOP] Early stopping triggered at epoch {ep+PHASE1_EPOCHS} due to validation mIoU plateau.")
            break

## 6. Training Curves & Visual Predictions

In [ ]:
# Plot Training Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train Loss", color="blue", lw=2)
ax1.plot(history["val_loss"], label="Val Loss", color="red", lw=2)
ax1.set_title("Combined BCE + Multiclass Dice Loss")
ax1.set_xlabel("Epoch")
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend()

ax2.plot(history["val_mIoU"], label="Validation mIoU", color="green", lw=2)
ax2.plot(history["val_pixel_acc"], label="Pixel Accuracy", color="purple", lw=2)
ax2.set_title("Validation Quality Metrics")
ax2.set_xlabel("Epoch")
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend()
plt.tight_layout()
plt.show()